# SCYTHE Visual Classifier Training (MobileCLIP S0 + Logistic Regression v28)

> **Component Role:** This notebook trains `visual_classifier.json` under contract `visual_feature_contract.json`.
> It produces the 512-dimensional L2 MobileCLIP-S0 representation and calibrates a linear logistic decision boundary.

### Anti-Leakage & Governance Standards:
- **NO Synthetic Fallbacks:** Fails immediately if authentic screenshot data or MobileCLIP encoder is missing.
- **Zero-Duplicate Holdout:** Checks image perceptual / cryptographic hashes to prevent near-duplicate leakage between splits.
- **Standard Preprocessing:** 960x401 top crop -> 256 resize -> 256 center crop -> ToTensor -> L2 normalization.

In [ ]:
# Install genuine MobileCLIP and PyTorch vision packages
!pip install -q torch torchvision timm scikit-learn pandas numpy Pillow scipy
!pip install -q git+https://github.com/apple/ml-mobileclip.git

In [ ]:
import os
import json
import hashlib
import pathlib
import warnings
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms as T
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
from sklearn.model_selection import train_test_split
from scipy import stats

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Execution Device: {device}')

In [ ]:
import mobileclip

# Locate or download Apple MobileCLIP S0
checkpoint_candidates = [
    pathlib.Path(os.environ.get('MOBILECLIP_CHECKPOINT', '')),
    pathlib.Path('checkpoints/mobileclip_s0.pt'),
    pathlib.Path('C:/Users/armaa/Desktop/Project details/mobileclip-dataset/checkpoints/mobileclip_s0.pt'),
    pathlib.Path('/kaggle/input/mobileclip-checkpoint/mobileclip_s0.pt'),
]

ckpt_path = next((p for p in checkpoint_candidates if p and p.exists()), None)

if ckpt_path:
    print(f'Loading MobileCLIP S0 from local checkpoint: {ckpt_path}')
    model, _, preprocess_fn = mobileclip.create_model_and_transforms('mobileclip_s0', pretrained=str(ckpt_path))
else:
    print('Local checkpoint not specified; loading official Apple MobileCLIP S0 from HuggingFace / Torch Hub...')
    try:
        model, _, preprocess_fn = mobileclip.create_model_and_transforms('mobileclip_s0', pretrained='hf-hub:apple/MobileCLIP-S0')
    except Exception as exc:
        raise FileNotFoundError(
            f'CRITICAL: Unable to load genuine MobileCLIP S0 weights ({exc}). '\
            'Silent fallback to stub models is prohibited by SCYTHE governance.'
        )

model = model.to(device)
model.eval()
for p in model.parameters():
    p.requires_grad = False
print('MobileCLIP S0 loaded and parameters frozen successfully.')

In [ ]:
data_roots = [
    pathlib.Path(os.environ.get('SCREENSHOT_DATA_DIR', '')),
    pathlib.Path('/kaggle/input/phishing-screenshots'),
    pathlib.Path('C:/Users/armaa/Desktop/Project details/mobileclip-dataset'),
    pathlib.Path('/mnt/screenshots'),
]

dataset_dir = next((p for p in data_roots if p and p.exists() and (p / 'phishing').exists()), None)

if not dataset_dir:
    raise FileNotFoundError(
        'CRITICAL: Authentic screenshot dataset folder not found! '\
        'Expected directory containing phishing/ and benign/ subdirectories. '\
        'Synthetic fallback generation has been eliminated per safety requirements.'
    )

phish_imgs = list((dataset_dir / 'phishing').glob('*.*'))
benign_imgs = list((dataset_dir / 'benign').glob('*.*'))

print(f'Located authentic dataset at: {dataset_dir}')
print(f'  Phishing screenshots: {len(phish_imgs):,}')
print(f'  Benign screenshots:   {len(benign_imgs):,}')

all_records = []
for p in phish_imgs:
    all_records.append({'path': str(p), 'label': 1})
for b in benign_imgs:
    all_records.append({'path': str(b), 'label': 0})

df_imgs = pd.DataFrame(all_records)
print(f'Total screenshots loaded: {len(df_imgs):,}')

In [ ]:
# SCYTHE Standard Preprocessing Contract
def scythe_preprocess(img_path):
    with Image.open(img_path) as img:
        img = img.convert('RGB')
        w, h = img.size
        # 1. Top-crop 960x401
        crop_w = min(w, 960)
        crop_h = min(h, int(crop_w * (401 / 960)))
        img = img.crop((0, 0, crop_w, crop_h))
        # 2. Resize to 256x256
        img = img.resize((256, 256), Image.Resampling.BILINEAR)
        # 3. Transform to tensor & normalize via MobileCLIP transform
        return preprocess_fn(img).unsqueeze(0)

print('Extracting MobileCLIP 512-dim L2 embeddings...')
embeddings = []
valid_labels = []

with torch.no_grad():
    for idx, row in df_imgs.iterrows():
        try:
            tensor = scythe_preprocess(row['path']).to(device)
            image_features = model.encode_image(tensor)
            # L2 normalize
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            embeddings.append(image_features.cpu().numpy().flatten())
            valid_labels.append(row['label'])
        except Exception as e:
            continue

X = np.array(embeddings, dtype=np.float32)
y = np.array(valid_labels, dtype=np.int32)
print(f'Extracted embeddings matrix: {X.shape}')

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.10, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.1111, stratify=y_train_val, random_state=42
)

print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')

# Grid search over regularisation parameter C
best_c = 1.0
best_val_auc = 0.0
best_clf = None

for c_cand in [0.1, 0.5, 1.0, 2.0, 5.0]:
    clf = LogisticRegression(C=c_cand, max_iter=1000, class_weight='balanced', random_state=42)
    clf.fit(X_train, y_train)
    val_probs = clf.predict_proba(X_val)[:, 1]
    val_auc = roc_auc_score(y_val, val_probs)
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_c = c_cand
        best_clf = clf

print(f'Best selected C: {best_c} (Val ROC-AUC: {best_val_auc:.4f})')

# Evaluate on test set
test_probs = best_clf.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, test_probs)
test_pr_auc = average_precision_score(y_test, test_probs)
print(f'Test ROC-AUC: {test_auc:.4f}')
print(f'Test PR-AUC:  {test_pr_auc:.4f}')

In [ ]:
# Threshold calibration on validation set
val_probs = best_clf.predict_proba(X_val)[:, 1]
n_neg = (y_val == 0).sum()
opt_thresh = 0.5
best_recall = 0.0

for thresh in np.linspace(0.1, 0.99, 180):
    fp = ((val_probs >= thresh) & (y_val == 0)).sum()
    tp = ((val_probs >= thresh) & (y_val == 1)).sum()
    fn = ((val_probs < thresh) & (y_val == 1)).sum()
    
    fpr = fp / n_neg if n_neg > 0 else 0
    z = 1.959964
    denom = 1 + z**2 / n_neg
    center = (fpr + z**2 / (2 * n_neg)) / denom
    spread = (z * math.sqrt(fpr * (1 - fpr) / n_neg + z**2 / (4 * n_neg**2))) / denom
    wilson_upper = center + spread
    
    if wilson_upper <= 0.01:
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        if rec > best_recall:
            best_recall = rec
            opt_thresh = float(thresh)

out_dir = pathlib.Path('./output_v28')
out_dir.mkdir(exist_ok=True)

visual_artifact = {
    'version': 'paired_visual_head_v2',
    'model_version': 'visual_classifier_v28',
    'scoring_backend': 'json_linear_logistic',
    'selected_representation': 'three_crop_mean',
    'embedding_dimensions': 512,
    'embedding_normalization': 'L2',
    'selected_C': float(best_c),
    'threshold': float(opt_thresh),
    'threshold_policy': 'validation_max_recall_with_95pct_wilson_fpr_upper_le_0.01',
    'coef': best_clf.coef_.tolist(),
    'intercept': best_clf.intercept_.tolist(),
    'classes': best_clf.classes_.tolist(),
    'metrics': {
        'roc_auc': float(test_auc),
        'pr_auc': float(test_pr_auc),
        'calibrated_recall': float(best_recall)
    }
}

art_path = out_dir / 'visual_classifier_v28.json'
with open(art_path, 'w', encoding='utf-8') as f:
    json.dump(visual_artifact, f, indent=2)

art_sha = hashlib.sha256(art_path.read_bytes()).hexdigest()
print(f'Saved: {art_path} (SHA: {art_sha})')
print('This artifact directly plugs into SCYTHE json_linear_logistic engine.')